# 第108章 K-Means客户分群实战

<!-- module-learning-arc:start -->
> **机器学习 模块主线｜第 23 / 34 步：深入客户分群与降维表达**
>
> **持续应用背景：** 建设可信预测系统：从统一训练流程开始，比较模型、处理不平衡、选择阈值、解释结果并保存完整 Pipeline，最终回答模型能否安全投入使用。
>
> **承接上一阶段：** 概率校准与Brier Score  →  **本章任务：** K-Means客户分群实战  →  **下一步：** 层次聚类与DBSCAN
>
> **大作业连接：** 本章练习将成为《模型上线评审会》的一部分，最终需要把候选模型变成经过预测合同、泄漏审计、业务阈值、错误分析和模型卡检查的上线建议。
<!-- module-learning-arc:end -->


## 本章场景

做客户运营时，常面对成百上千条交易流水，却说不清哪些客户高价值、该重点维护，哪些已快流失、需要用券挽回。


## 本章目标

学完本章，你将能够：

- **理解**：理解「K-Means客户分群实战」的核心思想、适用场景、关键假设与要解释的业务问题。
- **操作**：能按标准流程完成数据准备、模型训练与评估，并解读「K-Means客户分群实战」的关键输出指标。
- **迁移**：能把「K-Means客户分群实战」迁移到一份新数据上，独立完成任务并就结果给出有分寸的结论。


## 核心概念

**背景引入**：做客户运营时，常面对成百上千条交易流水，却说不清哪些客户高价值、该重点维护，哪些已快流失、需要用券挽回。K-Means 把「最近一次购物距今多久、一共买过几次、总共花了多少钱」压缩成客户画像并自动分组，让你不用硬翻表格也能抓住客户结构。

- 目标：\(\min_\mu\sum_i\min_k||x_i-\mu_k||^2\)
- 簇编号没有大小含义
- 轮廓系数兼顾簇内紧密和簇间分离
- 分群稳定性比单次最优分数更重要（打个比方：把客户分成几组，组号本身没有大小含义，关键看分得稳不稳——换个起点分组就全乱，那这“画像”就不可信。）


## 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 数据与问题定义 | `pd.read_csv()`、`sales.CustomerID.notna()`、`sales.InvoiceDate.max()`、`pd.Timedelta()` | 先明确样本、特征、目标和验证方式，再训练模型。 | 在交易行而非客户粒度聚类 |
| 模型、公式与诊断 | `np.log1p()`、`scores.items()`、`rfm.groupby()`、`.fit_transform()` | 把核心数学量映射到 sklearn 输出，并检查泛化表现。 | 金额偏态不处理 |


## 例 1｜数据与问题定义

先明确样本、特征、目标和验证方式，再训练模型。


<!-- math-foundation:chapter-108 -->
### 数学推导｜K-Means 最小化簇内平方距离

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜固定中心，分配样本。** $c_i=\arg\min_k\lVert x_i-\mu_k\rVert_2^2$。

**第 2 步｜固定分配，更新中心。** 对第 $k$ 个簇，平方距离和关于 $\mu_k$ 的最小值在样本均值处：

$$
\mu_k=\frac{1}{|C_k|}\sum_{x_i\in C_k}x_i
$$

**第 3 步｜交替执行。** 分配步和更新步都不会增大目标函数，因此算法会收敛到一个局部最优；不同初始中心可能得到不同结果。

**把上面的关系收束为本章计算式：**

$$
\min_{C_1,\ldots,C_K}\sum_{k=1}^{K}\sum_{x_i\in C_k}\lVert x_i-\mu_k\rVert_2^2
$$

**符号解释：** $\mu_k$ 是第 $k$ 个簇中心。

**代码对应：** 标准化特征，固定 `random_state`，结合 inertia、轮廓系数和业务可解释性选择 $K$。

**使用边界：** K-Means 偏好球状、大小相近的簇，簇编号本身没有顺序含义。


In [ ]:
import pandas as pd

sales = pd.read_csv(
    "/datasets/uci_online_retail_200k.csv", parse_dates=["InvoiceDate"]
)
sales = sales[
    (sales.Quantity > 0) & (sales.UnitPrice > 0) & sales.CustomerID.notna()
].copy()
sales["revenue"] = sales.Quantity * sales.UnitPrice
snapshot = sales.InvoiceDate.max() + pd.Timedelta(days=1)
rfm = (
    sales.groupby("CustomerID")
    .agg(
        recency=("InvoiceDate", lambda x: (snapshot - x.max()).days),
        frequency=("InvoiceNo", "nunique"),
        monetary=("revenue", "sum"),
    )
    .clip(lower=0)
)


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


**练一练**：改一个「数据字段或参数」，观察客户级 RFM 输入怎么变。

示例用 `snapshot = sales.InvoiceDate.max() + pd.Timedelta(days=1)` 定义「数据截止日」，并据此算出每个客户的 `recency`（最近一次购物距今的天数）。在下方把表示回看窗口的常量从 1 改成 30，重新得到 `rfm2`，对比 `recency` 的中位数与示例结果，再用一句话说明为什么所有客户的 `recency` 会一起变化。


In [ ]:
try:
    # 请在下方填写代码。示例中 rfm 是客户级表，snapshot 是数据截止日。
    # 任务：把回看窗口常量 offset_days 从 1 改为 30，并让 rfm2 据此重新计算 recency。
    offset_days = 1  # 改成 30，观察 recency 中位数怎么变
    # 请在下方填写代码：基于 snapshot + Timedelta(days=offset_days)
    # 重新计算每个客户的 recency（提示：用 groupby + (快照-最近一次购物).days）
    # TODO：请在下方完成 —— 练一练：改一个「数据字段或参数」，观察客户级 RFM 输入怎么变。 示例用 snapshot =
    # sales.Invoi

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 例 2｜模型、公式与诊断

把核心数学量映射到 sklearn 输出，并检查泛化表现。


In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

features = np.log1p(rfm)
Xs = StandardScaler().fit_transform(features)
scores = {}
models = {}
for k in range(2, 7):
    models[k] = KMeans(n_clusters=k, n_init=20, random_state=97).fit(Xs)
    scores[k] = silhouette_score(Xs, models[k].labels_)
best_k = max(scores, key=scores.get)
rfm["cluster"] = models[best_k].labels_
print("scores:", {k: round(v, 3) for k, v in scores.items()})
display(
    rfm.groupby("cluster")
    .agg(
        customers=("monetary", "size"),
        recency=("recency", "median"),
        frequency=("frequency", "median"),
        monetary=("monetary", "median"),
    )
    .round(1)
)


## 独立迁移练习

在不改变数据切分和指标的前提下，比较基线与一个模型设置。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    # TODO: 在此粘贴或改写最接近的示例。
    # 记录：我改了什么？预期会发生什么？实际观察到什么？
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(
        {"修改": change_note, "预期": expected_change, "观察": observed_change}
    )

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 本章实训：模型与基线比较

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

X = pd.DataFrame(
    {"visits": [1, 2, 3, 4, 5, 6], "discount": [0, 0, 1, 1, 1, 2]}
)
y = np.array([12, 15, 19, 23, 27, 31])
baseline = DummyRegressor(strategy="mean").fit(X, y)
model = LinearRegression().fit(X, y)
print("基线预测：", np.round(baseline.predict(X[:2]), 2))
print("模型预测：", np.round(model.predict(X[:2]), 2))
print("基线MAE：", round(mean_absolute_error(y, baseline.predict(X)), 2))
print("模型MAE：", round(mean_absolute_error(y, model.predict(X)), 2))


### 第一个结果怎么读

复杂模型之前先建立基线。只有在同一数据切分和同一指标下超过基线，模型才值得继续分析。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
X_changed = X.copy()
X_changed["visits"] = X_changed["visits"] + 1
changed_prediction = model.predict(X_changed)
print("原始前2个预测：", np.round(model.predict(X[:2]), 2))
print("访问次数+1后的预测：", np.round(changed_prediction[:2], 2))
print("预测变化：", np.round(changed_prediction[:2] - model.predict(X[:2]), 2))


### 第二个结果怎么读

只把一个特征整体加 1，观察预测变化。这个实验只能说明模型的预测响应，不能直接证明真实世界的因果关系。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：模型特征泄漏怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

data = pd.DataFrame(
    {
        "visits": [2, 4, 6],
        "duration_after_call": [30, 80, 120],
        "target": [0, 1, 1],
    }
)
forbidden = {"target", "duration_after_call"}
_demo_features = [column for column in data.columns if column not in forbidden]
print("禁止使用：", sorted(forbidden))
print("安全特征：", _demo_features)
print("原因：特征必须在预测时点已经可获得。")


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

如果一个字段在结果发生之后才产生，它即使与目标高度相关，也不能作为预测特征。先定义预测时点，再列可用字段。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- 在交易行而非客户粒度聚类
- 金额偏态不处理
- 把簇编号写成价值等级
- 没有验证不同随机种子下的稳定性


## 练习与作业

1. 修改一个关键参数并重新运行
2. 记录指标变化并解释原因
3. 检查结论是否依赖测试集或隐藏泄漏

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 108.11 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“修改一个关键参数并重新运行”。
2. **独立完成**：不复制示例代码，完成“记录指标变化并解释原因”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“检查结论是否依赖测试集或隐藏泄漏”，用一两句话说明你修改了什么。

### 108.11.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 108.11.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


## 小结

将 K-Means 用于客户分群，结合轮廓系数、簇规模和业务画像解释分群。


### 你已经掌握

- 构造客户级 RFM 特征
- 标准化后聚类
- 比较多个簇数
- 输出可行动的簇画像


### 需要注意

- 在交易行而非客户粒度聚类
- 金额偏态不处理
- 把簇编号写成价值等级
- 没有验证不同随机种子下的稳定性


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
practice_sizes = rfm.cluster.value_counts()
print(practice_sizes.sort_index())
